# EV Fleet Monitoring System — Day 1 Pipeline

This notebook walks through **everything** end-to-end, step by step, so the whole team can see what's happening at each stage:

1. Load the raw dataset
2. Explore it (EDA)
3. Clean it and engineer new columns
4. Save `Processed_EV_Data.csv` (the file every dashboard reads from)
5. Train the range-prediction model — **first the original 3-feature version, then an improved version**, so you can see exactly why accuracy jumped
6. Save `range_predictor.pkl`
7. Explain how `auth.py` / `app.py` login works and how to run the actual app

**Note on scope:** `app.py`, `auth.py`, `admin_dashboard.py`, `driver_dashboard.py` are Streamlit apps — Streamlit needs a browser + server, so those stay as `.py` files, they **cannot** run inside a notebook cell. Everything in the data/model pipeline (steps 1–6) runs fully in this notebook, and the same logic is mirrored into `src/data_prep.py` and `src/train_model.py` so those two `.py` files stay in sync with what you see here.

## Step 1: Load the raw dataset

Reading the original Excel file exactly as the team collected it — no changes yet.

In [10]:
import pandas as pd
import numpy as np
import os

RAW_PATH = os.path.join("..", "data", "raw", "ev_fleet_single_dataset.xlsx")
PROCESSED_PATH = os.path.join("..", "data", "processed", "Processed_EV_Data.csv")
MODEL_PATH = os.path.join("..", "src", "models", "range_predictor.pkl")

df_raw = pd.read_excel(RAW_PATH)
print(f"Shape: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns")
df_raw.head()

Shape: 10000 rows x 27 columns


,Record_ID,Date,Time,Car_ID,Company,Model,Driver_ID,Driver_Name,Driving_Style,Driving_Mode,...,Area,Latitude,Longitude,Electricity_Cost_INR,Revenue_INR,Exact_Maintenance_Cost_INR,Est_Full_Range_km,Current_Status,Live_Car_Status,Maintenance_Score
0,1,2025-06-29,08:22:00,TAT-01,Tata,Nexon EV,DRV-001,Amit,Rash,Highway,...,Adajan,21.16682,72.87282,84.11,343.78,11032.13,122.98,Working,Running,64
1,2,2025-05-24,21:00:00,TAT-01,Tata,Nexon EV,DRV-001,Amit,Rash,City,...,Katargam,21.18312,72.83251,107.31,855.45,11032.13,179.02,Working,Charging,64
2,3,2025-01-07,17:05:00,TAT-01,Tata,Nexon EV,DRV-001,Amit,Rash,City,...,Vesu,21.14495,72.78228,52.35,463.36,11032.13,187.88,Working,Running,64
3,4,2025-06-27,14:17:00,TAT-01,Tata,Nexon EV,DRV-001,Amit,Rash,Highway,...,Katargam,21.13944,72.85030,136.06,624.60,11032.13,138.36,Working,Idle,64
4,5,2025-06-29,12:00:00,TAT-01,Tata,Nexon EV,DRV-001,Amit,Rash,City,...,Varachha,21.15715,72.78147,47.10,298.40,11032.13,194.04,Working,Running,64


## Step 2: Explore the raw data (EDA)

Before cleaning anything, let's understand what we're working with — column types, missing values, and the fleet's basic shape.

In [11]:
print("Columns:", list(df_raw.columns))
print()
print("Data types:")
print(df_raw.dtypes)

Columns: ['Record_ID', 'Date', 'Time', 'Car_ID', 'Company', 'Model', 'Driver_ID', 'Driver_Name', 'Driving_Style', 'Driving_Mode', 'Speed_kmph', 'Distance_km', 'Battery_Capacity_kWh', 'Battery_Pct_Used', 'Harsh_Events', 'Location', 'City', 'Area', 'Latitude', 'Longitude', 'Electricity_Cost_INR', 'Revenue_INR', 'Exact_Maintenance_Cost_INR', 'Est_Full_Range_km', 'Current_Status', 'Live_Car_Status', 'Maintenance_Score']

Data types:
Record_ID                              int64
Date                          datetime64[ns]
Time                                  object
Car_ID                                object
Company                               object
Model                                 object
Driver_ID                             object
Driver_Name                           object
Driving_Style                         object
Driving_Mode                          object
Speed_kmph                           float64
Distance_km                          float64
Battery_Capacity_kWh       

In [12]:
print("Missing values per column:")
missing = df_raw.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values — clean dataset.")

Missing values per column:
No missing values — clean dataset.


In [13]:
print(f"Unique cars: {df_raw['Car_ID'].nunique()}")
print(f"Unique drivers: {df_raw['Driver_ID'].nunique()}")
print(f"Records per car: {df_raw.groupby('Car_ID').size().describe()}")
print()
print(f"Date range: {df_raw['Date'].min()} to {df_raw['Date'].max()}")

Unique cars: 50
Unique drivers: 50
Records per car: count     50.0
mean     200.0
std        0.0
min      200.0
25%      200.0
50%      200.0
75%      200.0
max      200.0
dtype: float64

Date range: 2025-01-01 00:00:00 to 2025-08-28 00:00:00


In [14]:
# The dataset has TWO status columns that can disagree on the same row —
# important to understand before we pick one as the source of truth.
print("Current_Status values:", df_raw['Current_Status'].unique())
print("Live_Car_Status values:", df_raw['Live_Car_Status'].unique())
print()
print("Live_Car_Status distribution:")
print(df_raw['Live_Car_Status'].value_counts())

Current_Status values: ['Working' 'Running' 'In Garage']
Live_Car_Status values: ['Running' 'Charging' 'Idle' 'Under Maintenance']

Live_Car_Status distribution:
Live_Car_Status
Running              6943
Charging             1330
Idle                 1211
Under Maintenance     516
Name: count, dtype: int64


In [15]:
df_raw[['Speed_kmph','Distance_km','Battery_Capacity_kWh','Battery_Pct_Used',
        'Harsh_Events','Electricity_Cost_INR','Revenue_INR',
        'Exact_Maintenance_Cost_INR','Est_Full_Range_km','Maintenance_Score']].describe()

,Speed_kmph,Distance_km,Battery_Capacity_kWh,Battery_Pct_Used,Harsh_Events,Electricity_Cost_INR,Revenue_INR,Exact_Maintenance_Cost_INR,Est_Full_Range_km,Maintenance_Score
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000,10000.000000
mean,67.547380,41.385498,44.996000,18.539780,0.783800,66.701533,475.871570,9304.551000,263.25590,50.240000
std,22.263284,26.974791,10.190229,16.431157,1.375953,54.792447,317.049794,4747.340309,78.38859,32.738503
min,9.100000,1.600000,17.200000,0.560000,0.000000,1.900000,19.340000,2157.530000,58.70000,1.000000
25%,50.300000,21.277500,38.600000,7.430000,0.000000,28.610000,239.397500,4853.880000,205.25000,19.000000
50%,64.700000,36.970000,45.350000,13.885000,0.000000,51.470000,419.865000,8372.795000,262.07500,45.500000
75%,81.800000,56.552500,52.300000,24.150000,1.000000,88.917500,649.012500,14467.560000,317.73000,85.000000
max,140.000000,240.880000,63.600000,100.000000,11.000000,565.400000,2996.730000,16636.850000,440.54000,100.000000


## Step 3: Clean and engineer features

Three changes, each with a reason:

1. **`Battery_Remaining_Percent`** — the raw data only has `Battery_Pct_Used` (% used). Our plan and the driver dashboard need % *remaining*, so we derive it: `100 - Battery_Pct_Used`.
2. **`Fleet_Status`** — decided to use `Live_Car_Status` (4 values: Running/Charging/Idle/Under Maintenance) as the single source of truth instead of `Current_Status` (3 values), since it's more granular and matches the admin dashboard's Running/Garage/Charging/Idle concept.
3. **`Month` / `Month_Num`** — split out of `Date` so the admin dashboard's month filter is a simple dropdown instead of parsing dates every time.

In [16]:
df = df_raw.copy()

# 1. Battery remaining
df["Battery_Remaining_Percent"] = (100 - df["Battery_Pct_Used"]).round(2)

# 2. Single source of truth for fleet status
df["Fleet_Status"] = df["Live_Car_Status"]

# 3. Date parts for the month filter
df["Date"] = pd.to_datetime(df["Date"])
df["Month"] = df["Date"].dt.strftime("%b")
df["Month_Num"] = df["Date"].dt.month

df[["Battery_Pct_Used","Battery_Remaining_Percent","Live_Car_Status","Fleet_Status","Date","Month","Month_Num"]].head()

,Battery_Pct_Used,Battery_Remaining_Percent,Live_Car_Status,Fleet_Status,Date,Month,Month_Num
0,28.76,71.24,Running,Running,2025-06-29,Jun,6
1,36.70,63.30,Charging,Charging,2025-05-24,May,5
2,17.90,82.10,Running,Running,2025-01-07,Jan,1
3,46.53,53.47,Idle,Idle,2025-06-27,Jun,6
4,16.11,83.89,Running,Running,2025-06-29,Jun,6


In [17]:
# Validate before saving — fail loud here rather than let a bad file
# quietly break every dashboard downstream.
assert df["Car_ID"].nunique() == 50, f"Expected 50 cars, got {df['Car_ID'].nunique()}"
assert df.isnull().sum().sum() == 0, "Unexpected missing values after enrichment"
assert df["Battery_Remaining_Percent"].between(0, 100).all(), "Battery_Remaining_Percent out of range"
print("All validation checks passed.")

All validation checks passed.


## Step 4: Save `Processed_EV_Data.csv`

This is the file every dashboard (`admin_dashboard.py`, `driver_dashboard.py`) and the training script read from. **Nobody should hand-edit this file** — if the schema needs to change, change the cleaning logic above (and in `src/data_prep.py`) and regenerate it.

In [18]:
os.makedirs(os.path.dirname(PROCESSED_PATH), exist_ok=True)
df.to_csv(PROCESSED_PATH, index=False)
print(f"Saved {len(df)} rows x {len(df.columns)} cols -> {PROCESSED_PATH}")

Saved 10000 rows x 31 cols -> ..\data\processed\Processed_EV_Data.csv


## Step 5: Train the range-prediction model

### 5a. First attempt — the original 3-feature plan

The original plan specified: `Battery_Remaining_Percent`, `Speed_kmph`, `Driving_Mode` → predict `Est_Full_Range_km`. Let's train that first and see how it does.

In [19]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

TARGET = "Est_Full_Range_km"

# --- Version A: original 3 features ---
features_a = ["Battery_Remaining_Percent", "Speed_kmph", "Driving_Mode"]
X_a = pd.get_dummies(df[features_a], columns=["Driving_Mode"], drop_first=True)
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X_a, y, test_size=0.2, random_state=42)

model_a = RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)
model_a.fit(X_train, y_train)
preds_a = model_a.predict(X_test)

mae_a = mean_absolute_error(y_test, preds_a)
r2_a = r2_score(y_test, preds_a)
print(f"Version A (original 3 features)")
print(f"  Test MAE: {mae_a:.2f} km")
print(f"  Test R^2: {r2_a:.3f}")

Version A (original 3 features)
  Test MAE: 43.47 km
  Test R^2: 0.497


**This matches what you saw when you ran `train_model.py`: MAE ~43 km, R² ~0.497.**

R² of 0.497 means the model only explains about half the variation in range — the other half is coming from something the model can't see with just these 3 features. MAE of 43 km on an average range of ~263 km is a fairly wide margin for a "predict range" feature.

### 5b. Why it's weak — checking what's missing

Every EV in this fleet has a different `Battery_Capacity_kWh` (battery *size*, not % charged) depending on the car model — a Tata Nexon EV and a Hyundai Ioniq 5 have very different total range even at the same battery %. The original 3 features never tell the model which car it's looking at, so it's guessing blind on that. Let's check the correlation.

In [20]:
df[["Battery_Capacity_kWh","Distance_km","Harsh_Events","Battery_Remaining_Percent",
    "Speed_kmph","Est_Full_Range_km"]].corr(numeric_only=True)["Est_Full_Range_km"].sort_values(ascending=False)

Est_Full_Range_km            1.000000
Battery_Capacity_kWh         0.745633
Battery_Remaining_Percent    0.576260
Harsh_Events                -0.134897
Distance_km                 -0.277554
Speed_kmph                  -0.570208
Name: Est_Full_Range_km, dtype: float64

### 5c. Version B — improved feature set

Adding `Battery_Capacity_kWh` (car's actual battery size), plus `Distance_km` and `Harsh_Events` (trip behavior signals already in the dataset). All three are legitimate inputs available at prediction time — nothing here leaks the target.

In [21]:
# --- Version B: improved features ---
features_b = ["Battery_Remaining_Percent", "Speed_kmph", "Driving_Mode",
              "Distance_km", "Harsh_Events", "Battery_Capacity_kWh"]
X_b = pd.get_dummies(df[features_b], columns=["Driving_Mode"], drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(X_b, y, test_size=0.2, random_state=42)

model_b = RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)
model_b.fit(X_train, y_train)
preds_b = model_b.predict(X_test)

mae_b = mean_absolute_error(y_test, preds_b)
r2_b = r2_score(y_test, preds_b)
print(f"Version B (+ Battery_Capacity_kWh, Distance_km, Harsh_Events)")
print(f"  Test MAE: {mae_b:.2f} km")
print(f"  Test R^2: {r2_b:.3f}")
print()
print(f"Improvement: MAE dropped {mae_a - mae_b:.2f} km, R^2 up {r2_b - r2_a:.3f}")

Version B (+ Battery_Capacity_kWh, Distance_km, Harsh_Events)
  Test MAE: 2.86 km
  Test R^2: 0.990

Improvement: MAE dropped 40.61 km, R^2 up 0.493


In [22]:
# Which features actually matter?
importance = pd.Series(model_b.feature_importances_, index=X_b.columns).sort_values(ascending=False)
print("Feature importance:")
print(importance)

Feature importance:
Battery_Capacity_kWh         0.649595
Speed_kmph                   0.291071
Driving_Mode_Highway         0.053092
Distance_km                  0.002850
Battery_Remaining_Percent    0.002378
Harsh_Events                 0.001014
dtype: float64


**`Battery_Capacity_kWh` alone explains most of it** — makes complete sense for an EV fleet: a car's total battery size is the single biggest determinant of its max range. The original plan's 3 features never told the model which car it was even looking at.

**Decision: use Version B as the production model.** MAE ~3 km and R² ~0.99 is a genuinely reliable predictor now — good enough to confidently show in the demo instead of a wide, hand-wavy estimate.

## Step 6: Save `range_predictor.pkl`

Saved as a bundle (`{model, feature_columns}`) rather than just the raw model — `driver_dashboard.py` needs the exact feature column order (including the one-hot encoded `Driving_Mode_Highway`) to build a matching input row at prediction time.

In [23]:
import joblib

os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
bundle = {"model": model_b, "feature_columns": list(X_b.columns)}
joblib.dump(bundle, MODEL_PATH)
print(f"Saved model bundle -> {MODEL_PATH}")
print(f"Feature columns (order matters): {list(X_b.columns)}")

Saved model bundle -> ..\src\models\range_predictor.pkl
Feature columns (order matters): ['Battery_Remaining_Percent', 'Speed_kmph', 'Distance_km', 'Harsh_Events', 'Battery_Capacity_kWh', 'Driving_Mode_Highway']


In [24]:
# Quick sanity check — predict range for a made-up car state
sample = pd.DataFrame([{
    "Battery_Remaining_Percent": 75.0,
    "Speed_kmph": 60.0,
    "Distance_km": 40.0,
    "Harsh_Events": 1,
    "Battery_Capacity_kWh": 40.0,
    "Driving_Mode_Highway": 1,
}])
sample = sample.reindex(columns=bundle["feature_columns"], fill_value=0)
predicted = bundle["model"].predict(sample)[0]
print(f"Sample prediction: {predicted:.1f} km")

Sample prediction: 235.9 km


## Step 7: How login works (`auth.py`) — and how to actually run the app

`auth.py` can't run inside this notebook (it needs Streamlit's browser session), but here's the logic in plain terms so it's not a black box:

```python
VALID_LOGINS = {
    "admin":  {"password": "admin123",  "role": "Admin"},
    "driver": {"password": "driver123", "role": "Driver"},
}
```

When someone logs in, `auth.py` checks the username against this dict, checks the password matches, and checks the selected role matches — if all three agree, `st.session_state.logged_in = True` and `app.py` routes to the matching dashboard.

### To actually see it running:

Open a terminal in the repo root (not inside `notebooks/`) and run:

```bash
streamlit run src/app.py
```

This opens a browser tab. Log in with:
- **Admin** → username `admin`, password `admin123`
- **Driver** → username `driver`, password `driver123`

### Keeping `train_model.py` in sync

This notebook is for understanding/experimenting. `src/train_model.py` is the "official" version the whole team runs — it's been updated to match Version B (the improved feature set) from this notebook, so running `python src/train_model.py` from the repo root now reproduces the ~3 km MAE result, not the old ~43 km one.